# Demo notebook for using event detection class with data in google drive
* 3/11/2025 setup GT

## cloning and installing a custom module

In [ ]:
!git clone https://github.com/GergelyTuri/sleep.git

In [ ]:
%cd sleep
!pip install .

## Imports

In [ ]:
from os.path import join
import matplotlib.pyplot as plt

# custom modul specific imports
from src.io.suite2p_io import Suite2p as s2p
from src.preprocessing import dfof

In [ ]:
from google.colab import drive
drive.mount('/gdrive')

In [ ]:
# pointer to data
sima_folder = "/gdrive/Shareddrives/Turi_lab/Data/Sleep/2p/Analysis/data/dock11b1/6_17/TSeries-06172024-0946-001/TSeries-06172024-0946-001.sima/"

In [ ]:
s2p_data = s2p(join(sima_folder, "suite2p"))
raw_signal = s2p_data.get_cells()
npil_signal = s2p_data.get_npil()
raw_signal.shape, npil_signal.shape

In [ ]:
type(raw_signal)

In [ ]:
# # example plot
# plt.plot(raw_signal[0])

In [ ]:
dfof_strategy = dfof.Suite2pDFOF()
dfof = dfof_strategy.calculate(raw_signal, npil_signal)

In [ ]:
# # example plot
# dfof.iloc[0].plot()

In [ ]:
# dfof.shape

In [ ]:
# dfof.head()

## Event detection class

In [ ]:
# Import the class
from src.preprocessing.event_detection import EventDetection, Denoiser
import numpy as np

In [ ]:
dfof.shape

In [ ]:
# Convert dfof into an array
dfof_array = dfof.to_numpy()

In [ ]:
# Initialize the transient analysis module
detection = EventDetection()

In [ ]:
# Initialize the denoiser with a specified method
denoiser = Denoiser(method="savgol", window_length=11, polyorder=3)

# Apply denoising
dfof_denoised = denoiser.apply(dfof_array)

In [ ]:
# Run transient detection iteratively
final_transients, final_noise, _ = detection.run_transient_detection(dfof_denoised, frame_period=0.1)

# Visualize results
detection.visualize_all(
    raw_signals=dfof_array,
    denoised_signals=dfof_denoised,
    noise=final_noise,
    transients=final_transients,
    cell_index=10 # change this to visualize other cells
)


In [ ]:
# Convert to DataFrame
df = detection.transients_to_dataframe(final_transients)

# Use percentiles to define new bounds (e.g., 10th and 90th)
min_dur = np.percentile(df['duration'], 10)

print(f"Suggested MIN_DURATION: {min_dur:.3f}")


In [ ]:
df.head(10)

In [ ]:
#Create new detector with new found min_duration
detection_refined = EventDetection(MIN_DURATION=3)

In [ ]:
# Run transient detection iteratively
final_transients, final_noise, _ = detection_refined.run_transient_detection(dfof_denoised, frame_period=0.1)

# Visualize results
detection_refined.visualize_all(
    raw_signals=dfof_array,
    denoised_signals=dfof_denoised,
    noise=final_noise,
    transients=final_transients,
    cell_index=10 # change this to visualize other cells
)


In [ ]:
import random

In [ ]:
# Check for 10 random traces

cell_indices = list(range(len(dfof_denoised)))

# Take 10 random ones (without replacement)
random_cells = random.sample(cell_indices, 10)

# Plot each one
for cell in random_cells:
    detection.plot_transient_events(
        dfof_signals=dfof_denoised,
        transients=final_transients,
        cell_index=cell
    )


In [ ]:
transients_df = detection.transients_to_dataframe(final_transients)
transients_df.head(10)


In [ ]:
# Optional export to csv
detection.export_transients_to_csv(final_transients, "my_transients.csv")

In [ ]:
# Optional export to html
detection.visualize_all(
    raw_signals=dfof_array,
    denoised_signals=dfof_denoised,
    noise=final_noise,
    transients=final_transients,
    cell_index=0, # change this to visualize other cells
    save_prefix="cell0"
)